# xAI cấp độ 1: giải thích kết quả xếp hạng của CLIP + Graph

**Yêu cầu của đề (cấp độ 1):** hiển thị quan hệ / đường đi được dùng khi xếp hạng cho một số truy vấn; phân tích 5 truy vấn thành công và 5 truy vấn thất bại.

Notebook này chạy toàn bộ pipeline rồi in kết quả:

1. Cài môi trường, lấy embedding CLIP, tải annotation, dựng đồ thị (`scripts/setup.py`).
2. Huấn luyện GNN và chọn `alpha` trên val (`scripts/train.py`).
3. **Quan hệ được dùng khi xếp hạng** cho vài câu tự gõ.
4. **5 ca thành công + 5 ca thất bại**, chọn theo quy tắc cố định, kèm giải thích và ảnh.
5. Lưu các ca ra Markdown để đưa vào báo cáo.

**Giải thích ở đây là gì?** Điểm cuối = `alpha · cos(q, ảnh) + (1 − alpha) · điểm đồ thị`. Điểm đồ thị do GNN một lớp tính từ các cạnh *khái niệm → ảnh* (ADR-0046). Với ảnh đứng đầu, Occlusion bỏ **từng** cạnh đó, chạy lại chính mô hình và đo điểm giảm bao nhiêu (ADR-0016). Cạnh làm điểm giảm nhiều nhất là quan hệ "đẩy ảnh lên". Số in ra là mức ảnh hưởng lên điểm cuối (đã nhân `1 − alpha`).

## Chạy trên Kaggle (khuyên dùng)

1. **Settings:** bật **Internet**, chọn **GPU** (T4/P100). CPU cũng chạy được, chậm hơn (khoảng 9 phút một epoch).
2. **Add Input:** output của notebook `qphulong/prepare-clip-embeddings-for-mmkg-xai`, bản `visual_genome` (embedding CLIP).
3. *(Tuỳ chọn, để xem ảnh)* **Add Input:** dataset `lyte69/visualgenome-part1-part2` (ảnh Visual Genome).
4. *(Tuỳ chọn, log W&B)* **Add-ons → Secrets:** `WANDB_API_KEY`.
5. **Save Version → Save & Run All** để giữ checkpoint và file kết quả trong Output.

**Colab:** giống vậy, nhưng phải tự đặt embedding vào `artifacts/features/visual_genome/ViT-B-32__laion2b_s34b_b79k/` (tải từ Kaggle hoặc chép từ Drive) và chép `artifacts/checkpoints/` sang Drive để không mất khi ngắt phiên.

**Máy local:** mở notebook trong `notebooks/` với `uv run --group ml --group graph --group demo jupyter lab`.

Hướng dẫn đầy đủ: `docs/guides/chay-pipeline.md`.

## 0. Cấu hình

In [ ]:
REPO_URL = "https://github.com/FluteInTheRain/graph-semantic-image-editing.git"
BRANCH = "feat/gnn-rerank"
SEEDS = [0]  # seeds to train; the cases below use the first one (3 seeds for the report table)
SPLIT = "test"  # only LOOK at test: change nothing after seeing it (ADR-0009); "val" while tuning
NUM_CASES = 5  # successes and failures, each
TYPED_QUERIES = [
    "a dog lying on a couch",
    "a man riding a horse on the beach",
    "a plate of food with a fork",
]
TOP_M = 3  # relations shown per side in an explanation

## 1. Môi trường và dữ liệu

In [ ]:
import glob
import os
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
ON_COLAB = "google.colab" in sys.modules

if Path("../src/mmxai").exists():  # opened from notebooks/ in a local clone
    os.chdir("..")
elif not Path("src/mmxai").exists():  # Kaggle / Colab: fetch the code
    if not Path("graph-semantic-image-editing").exists():
        !git clone -b {BRANCH} {REPO_URL}
    os.chdir("graph-semantic-image-editing")
    # torch is preinstalled on Kaggle / Colab: do not reinstall it, add only what is missing.
    !pip install -q -e . torch-geometric faiss-cpu open-clip-torch wandb
print("repo:", Path.cwd())

In [ ]:
def find_in_input(pattern, max_depth=4):
    """Paths under /kaggle/input matching pattern, searching a few levels only (fast)."""
    found = []
    for depth in range(max_depth + 1):
        found += glob.glob("/kaggle/input/" + "*/" * depth + pattern)
    return sorted(found)


# CLIP features: link the Kaggle input to the path the configs read.
FEATURES = Path("artifacts/features/visual_genome/ViT-B-32__laion2b_s34b_b79k")
if not (FEATURES / "meta.json").exists() and ON_KAGGLE:
    found = find_in_input("visual_genome/ViT-B-32__laion2b_s34b_b79k/meta.json")
    if found:
        FEATURES.parent.mkdir(parents=True, exist_ok=True)
        FEATURES.symlink_to(Path(found[0]).parent)
assert (FEATURES / "meta.json").exists(), (
    f"No CLIP features in {FEATURES}: add them as input (Kaggle) or copy them there"
)

# Visual Genome images (optional): only used to draw the cases.
IMAGE_DIRS = sorted(glob.glob("data/visual_genome/images/VG_100K*"))
if ON_KAGGLE:
    IMAGE_DIRS += find_in_input("VG_100K*")
print("features:", FEATURES.resolve())
print("image folders:", IMAGE_DIRS or "none (cases are shown as text only)")

# Weights & Biases (optional, ADR-0047): training curves on the cloud.
if ON_KAGGLE and "WANDB_API_KEY" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient

        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        print("no WANDB_API_KEY secret: training is not logged to W&B")

## 2. Tải annotation, dựng đồ thị (`scripts/setup.py`)

Tải `objects.json` (kiểm sha256) và `region_descriptions.json` nếu thiếu, rồi dựng đồ thị cạnh CLIP top-16. Bước nào đã xong thì bỏ qua.

In [ ]:
!python scripts/setup.py

## 3. Huấn luyện GNN và chọn `alpha` trên val (`scripts/train.py`)

Chỉ học trên ảnh train; early stopping theo MRR@100 trên val; `alpha` chọn trên val và lưu cạnh checkpoint. Seed đã xong thì bỏ qua.

In [ ]:
!python scripts/train.py --seeds {" ".join(str(seed) for seed in SEEDS)}

In [ ]:
import json

from mmxai.models.training import checkpoint_folder, load_chosen_alpha, load_gnn_config

config = load_gnn_config("configs/gnn_vg.yaml")
SEED = SEEDS[0]
meta = json.loads((checkpoint_folder(config, SEED) / "meta.json").read_text())
alpha = load_chosen_alpha(config, SEED)
print(f"seed {SEED}: best epoch {meta['best_epoch']}, alpha chosen on val = {alpha}")
for row in meta["history"]:
    print(row)

## 4. Quan hệ được dùng khi xếp hạng, cho vài câu tự gõ

Với mỗi câu: 5 ảnh đứng đầu sau khi xếp hạng lại, và các cạnh *khái niệm → ảnh* đã đẩy ảnh đứng đầu lên hoặc kéo xuống. Câu tự gõ không có đáp án nên không có hạng của ảnh đúng.

In [ ]:
from mmxai.explain.demo import cases_table, cases_to_markdown, format_result, run_demo

typed = run_demo(
    config,
    mode="clip_graph",
    seed=SEED,
    split_name=SPLIT,
    num_queries=0,
    typed_texts=TYPED_QUERIES,
    sample_seed=0,
    show_top=5,
    top_m=TOP_M,
    min_contribution=0.0,
    data_dir="data/visual_genome",
)
for number, result in enumerate(typed, start=1):
    print(format_result(result, number, "clip_graph"), end="\n\n")

## 5. Năm ca thành công và năm ca thất bại

**Quy tắc chọn (cố định trước khi xem kết quả, PDF §6):** xếp hạng mọi câu mô tả của split; **thành công** = ảnh đúng đứng hạng 1; **thất bại** = không có ảnh đúng trong top 10. Lấy 5 ca đầu tiên mỗi loại theo `text_id` tăng dần (`mmxai.explain.demo.pick_fixed_cases`).

Với mỗi ca, ngoài giải thích của ảnh đứng đầu, còn có giải thích của **ảnh nguồn** (ảnh mà câu mô tả được viết cho) khi nó không đứng đầu, và **hạng khi chỉ dùng CLIP** để thấy đồ thị đã đẩy ảnh đúng lên hay kéo xuống.

In [ ]:
import pandas as pd

cases = run_demo(
    config,
    mode="clip_graph",
    seed=SEED,
    split_name=SPLIT,
    num_queries=NUM_CASES,
    typed_texts=[],
    sample_seed=0,
    show_top=5,
    top_m=TOP_M,
    min_contribution=0.0,
    data_dir="data/visual_genome",
    pick="fixed",
)
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(cases_table(cases))

In [ ]:
from mmxai.explain.figures import show_case

for number, result in enumerate(cases, start=1):
    print(format_result(result, number, "clip_graph"))
    if IMAGE_DIRS:
        show_case(result, IMAGE_DIRS)
    print()

### Gợi ý phân tích (điền vào báo cáo)

**Ca thành công:**
- Các khái niệm "đẩy lên" có khớp với nội dung câu truy vấn không? (ví dụ câu có "dog" và cạnh `dog` đẩy lên)
- Hạng khi chỉ dùng CLIP có thấp hơn không, tức đồ thị có giúp, hay CLIP đã đúng sẵn?

**Ca thất bại:**
- Ảnh sai đứng đầu nhờ khái niệm nào? Khái niệm đó có trong câu không (nhầm vì khái niệm chung chung như `man`, `shirt`)?
- Ảnh nguồn có thiếu cạnh tới khái niệm chính của câu không? Cạnh lấy từ CLIP top-16 nên có thể sai (ADR-0046).
- Đồ thị kéo ảnh đúng xuống (hạng sau > hạng CLIP) hay CLIP đã sai từ đầu?
- Câu mô tả có mơ hồ không (rất ngắn, nhiều ảnh cùng hợp)?

**Giới hạn cần nói rõ:** giải thích chỉ gồm cạnh *khái niệm → ảnh* (một lớp, không có đường đi nhiều bước), và cạnh lấy từ CLIP nên không mang thông tin ngoài CLIP (ADR-0046, mục Hệ quả).

## 6. Lưu các ca ra Markdown cho báo cáo

In [ ]:
from datetime import datetime

out_dir = Path("results/runs") / f"{datetime.now():%Y%m%d-%H%M}_xai_level1_seed{SEED}"
out_dir.mkdir(parents=True, exist_ok=True)
header = f"# xAI level 1 cases\n\nsplit: {SPLIT}, seed: {SEED}, rule: rank 1 / not in top 10\n\n"
(out_dir / "cases.md").write_text(
    header
    + cases_to_markdown(typed, "typed queries")
    + "\n"
    + cases_to_markdown(cases, "clip_graph")
)
pd.DataFrame(cases_table(cases)).to_csv(out_dir / "cases.csv", index=False)
print("saved", out_dir)